# Week 3 v2 — Phase 3 compute-budget pilot

DESIGN.md v0.5 §8. This notebook times each component of the real run on T documents only and applies the pre-registered §8 selection rule to choose the compute settings (model size, precision, step count, BPE vocabulary) and whether a second char seed fits.

- It trains on T data only and scores nothing on N or F.
- `data.load_documents` reads the whole corpus, so N and F text is loaded, but only for the SHA-256 and body-length checks; it is dropped right after. It never reaches a tokenizer fit, a model or a statistic. N and F sizes come from the character counts in `split_manifest.json`.
- Pilot models are discarded. No loss or BPC is recorded.
- Output: `pilot/phase3_timing.json` (also printed at the end).

Run with Runtime → Run all on a Colab T4. It needs no manual steps. The notebook stops if CUDA is unavailable. Setting `PILOT_SMOKE=1` runs it locally on CPU with tiny sizes, to catch bugs only. Smoke numbers mean nothing.

## 1. Bootstrap

In [ ]:
import time

t_import0 = time.perf_counter()  # a cold "Run all" pays this in the real run
import os  # noqa: E402
import platform  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
from pathlib import Path  # noqa: E402

import torch  # noqa: E402

SMOKE = os.environ.get("PILOT_SMOKE") == "1"  # local bug-catching run only
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif SMOKE:
    DEVICE = torch.device("cpu")  # evaluate.py sums in float64: no MPS
else:
    raise RuntimeError("CUDA GPU required: Runtime -> Change runtime type "
                       "-> T4 GPU, then Run all.")
torch.zeros(1, device=DEVICE).sum().item()  # forces CUDA context init
import_cuda_s = time.perf_counter() - t_import0

REPO_URL = "https://github.com/FranQuant/the-ai-engineer.git"
BRANCH = "capstone/week03-v2"
SUBDIR = Path("capstones/week03_transformers")
MODULES = ("data.py", "bpe.py", "model.py", "evaluate.py")


def has_modules(p: Path) -> bool:
    return all((p / m).is_file() for m in MODULES)


cwd = Path.cwd()
candidates = [cwd.parent, cwd, cwd / SUBDIR, cwd / "the-ai-engineer" / SUBDIR]
W3 = next((p.resolve() for p in candidates if has_modules(p)), None)
clone_s = None  # None: modules already present, clone not measured
if W3 is None:
    dest = cwd / "the-ai-engineer"
    if dest.exists():
        raise RuntimeError(f"{dest} exists but lacks {MODULES}; remove it")
    t0 = time.perf_counter()
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(dest)], check=True)
    clone_s = time.perf_counter() - t0
    W3 = (dest / SUBDIR).resolve()
    if not has_modules(W3):
        raise RuntimeError(f"clone lacks {MODULES} under {SUBDIR}")
sys.path.insert(0, str(W3))

t0 = time.perf_counter()
import bpe  # noqa: E402
import data  # noqa: E402
import evaluate  # noqa: E402
import model  # noqa: E402
import_cuda_s += time.perf_counter() - t0  # excludes the clone

GIT_COMMIT = subprocess.run(["git", "-C", str(W3), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
GPU_NAME = (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
            else f"none ({DEVICE.type})")
ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "gpu": GPU_NAME, "device": DEVICE.type,
       "git_commit": GIT_COMMIT, "module_dir": str(W3), "smoke": SMOKE}
for k, v in ENV.items():
    print(f"{k:>10}: {v}")
print(f"imports + CUDA init: {import_cuda_s:.1f} s")
if clone_s is not None:
    print(f"clone: {clone_s:.1f} s")


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

In [ ]:
# Pilot settings. Smoke mode shrinks every size; its numbers are not used.
BLOCK_SIZE = 256
BATCH = 8 if SMOKE else 64
WARMUP_STEPS = 2 if SMOKE else 20
TIMED_STEPS = 3 if SMOKE else 50
VOCABS = (4000, 2000, 1000)  # largest first, the selection order
CONFIGS = {  # selection order C1, C2, C3
    "C1": dict(d_model=256, num_layers=6, num_heads=8, d_ff=1024),
    "C2": dict(d_model=192, num_layers=4, num_heads=6, d_ff=768),
    "C3": dict(d_model=128, num_layers=4, num_heads=4, d_ff=512),
}
PRECISIONS = ("fp16", "fp32")  # fp16 first within each config
TOKENIZERS = ("char", "bpe")
SCORE_CHARS_CAP = 20_000 if SMOKE else None  # None: the full N+F total
NGRAM_ORDER = 5
FIGURES_ESTIMATE_S = 30.0  # used for any overhead that is not measurable
# N monitoring in the real run: per training run, MON_EVALS evaluations of
# MON_BATCHES batches of MON_BATCH windows. Timed here on T windows.
MON_EVALS = 10
MON_BATCHES = 20
MON_BATCH = 8 if SMOKE else 64
MON_WARMUP = 2
# Bootstrap (§6): meeting resamples for each of H1, H2, H3.
BOOT_RESAMPLES = 2000
BOOT_HYPOTHESES = ("H1", "H2", "H3")

BUDGET_S = 720.0
MIN_STEPS = 1500
BPE_FIT_LIMIT_S = 90.0

PILOT_DIR = W3 / "pilot"
PILOT_DIR.mkdir(exist_ok=True)
OUT_PATH = Path(os.environ.get("PILOT_OUT", PILOT_DIR / (
    "phase3_timing.smoke.json" if SMOKE else "phase3_timing.json")))

## 2. Load documents (hash and length checks run), then T only

In [ ]:
t0 = time.perf_counter()
_all_docs = data.load_documents()  # SHA-256 checks; fails closed
load_s = time.perf_counter() - t0
t_docs = [d for d in _all_docs if d.split == "T"]
del _all_docs  # N and F text served the checks only; T only from here on

# N and F sizes: character counts from the split manifest only.
split_manifest = data.load_split_manifest()


def manifest_chars(splits):
    return sum(e["body_code_points"] for m in split_manifest["meetings"]
               if m["split"] in splits for e in m["documents"])


nf_chars = manifest_chars(("N", "F"))
f_chars = manifest_chars(("F",))
t_chars = sum(len(d.body) for d in t_docs)
print(f"load + hash checks: {load_s:.2f} s")
print(f"T: {len(t_docs)} documents, {t_chars:,} body characters")
print(f"N+F (manifest counts): {nf_chars:,} body characters")
print(f"F   (manifest counts): {f_chars:,} body characters")

## 3a. BPE and the vocabulary choice

Fit once at vocab 4000 and time it. The smaller vocabularies are prefixes of its merge list (bpe.py), so they come from `truncate`. Their fit time is at most the vocab-4000 fit time. If the 4000 fit exceeds the 90 s limit, the smaller vocabularies are fitted directly and timed.

The BPE vocabulary is fixed here, before any timing: the largest of 4000, 2000, 1000 whose shortest T document has at least block_size + 1 tokens (serialized) and whose fit time is at most 90 s. BPE training, monitoring and scoring below are all timed at this vocabulary.

In [ ]:
def fit_bpe(v):
    t0 = time.perf_counter()
    tok = bpe.SimpleBPE.from_documents(t_docs, v)
    return tok, time.perf_counter() - t0


bpe_full, fit_4000_s = fit_bpe(4000)
bpe_fit = {4000: {"fit_s": fit_4000_s, "measured": True}}
for v in VOCABS[1:]:
    if fit_4000_s <= BPE_FIT_LIMIT_S:
        bpe_fit[v] = {"fit_s": fit_4000_s, "measured": False,
                      "note": "upper bound: vocab-4000 fit time"}
    else:
        _, s = fit_bpe(v)
        bpe_fit[v] = {"fit_s": s, "measured": True}

bpe_stats = {}
for v in VOCABS:
    tok = bpe_full.truncate(v)
    body_lens = [len(tok.encode(d.body)) for d in t_docs]
    bpe_stats[v] = {
        "vocab_size": tok.vocab_size,
        "tokens_per_char": sum(body_lens) / t_chars,
        "shortest_doc_body_tokens": min(body_lens),
        # serialized = BOS + genre + body: what WindowSampler requires
        "shortest_doc_serialized_tokens": min(body_lens) + data.PREFIX_LEN,
        **bpe_fit[v],
    }
    s = bpe_stats[v]
    print(f"vocab {v}: {s['tokens_per_char']:.4f} tokens/char, shortest T "
          f"doc {s['shortest_doc_serialized_tokens']} tokens (serialized), "
          f"fit {s['fit_s']:.1f} s ({'measured' if s['measured'] else 'upper bound'})")

VOCAB_RULE = (f"BPE vocab = the largest of {{4000, 2000, 1000}} whose shortest "
              f"T document has >= block_size + 1 = {BLOCK_SIZE + 1} tokens "
              f"and whose fit time is <= {BPE_FIT_LIMIT_S:.0f} s.")
BPE_VOCAB = next((v for v in VOCABS
                  if bpe_stats[v]["shortest_doc_serialized_tokens"]
                  >= BLOCK_SIZE + 1
                  and bpe_stats[v]["fit_s"] <= BPE_FIT_LIMIT_S), None)
print(f"\n{VOCAB_RULE}\nBPE vocab: {BPE_VOCAB or 'none qualifies'}")
# No BPE vocab -> NO CONFIG FITS; only the char runs are still timed.
ACTIVE_TOKENIZERS = TOKENIZERS if BPE_VOCAB else ("char",)

## 3b. Training and N-monitoring throughput

Batch 64, 20 warm-up steps, 50 timed steps, each step synchronized and timed. A step is sampling, forward, backward, clipping, AdamW and the LR schedule, as in the real run. fp16 uses `torch.autocast` with a GradScaler. BPE uses the vocabulary chosen in 3a.

N monitoring is timed on the same model right after its training steps: eval mode, no gradients, the run's precision, batches of 64 T windows (N is never touched), 2 warm-up batches, then 20 timed batches. The cost of one run's monitoring is 10 evaluations × 20 batches × the median batch time.

In [ ]:
import math
import statistics

t0 = time.perf_counter()
char_vocab = data.CharVocab.from_documents(t_docs)
tokenizers = {"char": char_vocab}
if BPE_VOCAB:
    tokenizers["bpe"] = bpe_full.truncate(BPE_VOCAB)  # fresh cache
samplers = {k: data.WindowSampler.from_documents(t_docs, tok, BLOCK_SIZE)
            for k, tok in tokenizers.items()}
tokenize_s = time.perf_counter() - t0
print(f"tokenize T for training (char + BPE {BPE_VOCAB}): "
      f"{tokenize_s:.2f} s")


def build_model(cfg_name, tok):
    torch.manual_seed(0)
    cfg = model.ModelConfig(vocab_size=tok.vocab_size, block_size=BLOCK_SIZE,
                            **CONFIGS[cfg_name])
    return model.TinyTransformerLM(cfg).to(DEVICE)


def empty_cache():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def time_training(cfg_name, tok_name, precision):
    net = build_model(cfg_name, tokenizers[tok_name])
    n_params = sum(p.numel() for p in net.parameters())
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=0.1)
    total = WARMUP_STEPS + TIMED_STEPS
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(
        1.0, (s + 1) / 10) * 0.5 * (1 + math.cos(math.pi * s / total)))
    amp = precision == "fp16"
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=amp)
    gen = torch.Generator().manual_seed(0)
    sampler = samplers[tok_name]
    net.train()

    def step():
        x, y, _ = sampler.sample(BATCH, generator=gen)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=amp):
            _, loss = net(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()

    for _ in range(WARMUP_STEPS):
        step()
    sync()
    times = []
    for _ in range(TIMED_STEPS):
        t0 = time.perf_counter()
        step()
        sync()
        times.append(time.perf_counter() - t0)

    # N-monitoring proxy: same model and precision, T windows only.
    mon_gen = torch.Generator().manual_seed(1)
    net.eval()

    @torch.no_grad()
    def eval_batch():
        x, y, _ = sampler.sample(MON_BATCH, generator=mon_gen)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=amp):
            _, loss = net(x, y)
        loss.item()  # the real run logs the value

    for _ in range(MON_WARMUP):
        eval_batch()
    sync()
    mon_times = []
    for _ in range(MON_BATCHES):
        t0 = time.perf_counter()
        eval_batch()
        sync()
        mon_times.append(time.perf_counter() - t0)
    mon_batch_s = statistics.median(mon_times)
    del net, opt, scaler  # pilot models are discarded
    empty_cache()
    return {"s_per_step_median": statistics.median(times),
            "s_per_step_mean": statistics.fmean(times),
            "s_per_step_min": min(times), "s_per_step_max": max(times),
            "n_params": n_params, "timed_steps": TIMED_STEPS,
            "warmup_steps": WARMUP_STEPS, "batch": BATCH,
            "monitor_s_per_batch_median": mon_batch_s,
            "monitor_s_per_batch_max": max(mon_times),
            "monitor_run_s": MON_EVALS * MON_BATCHES * mon_batch_s}


throughput = {}
for c in CONFIGS:
    for tk in ACTIVE_TOKENIZERS:
        for p in PRECISIONS:
            r = time_training(c, tk, p)
            throughput[f"{c}/{tk}/{p}"] = r
            print(f"{c} {tk:>4} {p}: {r['s_per_step_median']*1e3:8.1f} ms/step "
                  f"median (mean {r['s_per_step_mean']*1e3:.1f}), "
                  f"{r['n_params']/1e6:.2f}M params; monitoring "
                  f"{r['monitor_run_s']:.1f} s/run")

## 3c. Scoring throughput

`evaluate.score_documents` on T documents, taken in load order until their characters reach the N+F total. Scoring runs as evaluate.py does it (fp32, batch 1, stride = block_size / 2), with freshly initialized models, because timing does not depend on the weights. Each model is first warmed up on the shortest T document (untimed, with a separate tokenizer instance so the BPE encode cache of the timed pass stays cold); then one timed pass.

Projection (§8): char scoring to N+F (the char model scores N and F); BPE scoring to F only (H3 uses F). The char F projection is kept for the seed-2 check.

In [ ]:
score_target = nf_chars if SCORE_CHARS_CAP is None else SCORE_CHARS_CAP
score_docs, score_chars = [], 0
for d in t_docs:
    if score_chars >= score_target:
        break
    score_docs.append(d)
    score_chars += len(d.body)
print(f"scoring set: {len(score_docs)} T documents, {score_chars:,} chars "
      f"(target {score_target:,})")

warm_doc = min(t_docs, key=lambda d: len(d.body))
SCORE_TARGETS = {"char": ("N+F", nf_chars), "bpe": ("F", f_chars)}


def fresh_tokenizer(tk):
    return char_vocab if tk == "char" else bpe_full.truncate(BPE_VOCAB)


scoring = {}
for c in CONFIGS:
    for tk in ACTIVE_TOKENIZERS:
        tok = fresh_tokenizer(tk)
        net = build_model(c, tok)
        evaluate.score_documents(net, fresh_tokenizer(tk), [warm_doc],
                                 BLOCK_SIZE)  # warm-up, untimed
        sync()
        t0 = time.perf_counter()
        scores = evaluate.score_documents(net, tok, score_docs, BLOCK_SIZE)
        sync()
        s = time.perf_counter() - t0
        assert len(scores) == len(score_docs)
        del net, scores
        empty_cache()
        per_100k = s / score_chars * 1e5
        target, target_chars = SCORE_TARGETS[tk]
        scoring[f"{c}/{tk}"] = {
            "s": s, "chars": score_chars, "s_per_100k_chars": per_100k,
            "projected_to": target,
            "projected_s": per_100k * target_chars / 1e5,
            "projected_f_s": per_100k * f_chars / 1e5,
            "warmup_doc": warm_doc.id, "timed_passes": 1}
        print(f"{c} {tk:>4}: {per_100k:6.2f} s/100k chars -> "
              f"{per_100k * target_chars / 1e5:6.1f} s projected for {target}")

## 3d. n-gram (Witten–Bell, order 5): timing stub

**Estimate.** The real n-gram is not written yet. This stub does the same work: interpolated Witten–Bell, order 5, uniform order-0 fallback over the T character vocabulary. It is fitted on all T bodies and scores the same T documents as 3c; the scoring time is projected to F (§8: the n-gram is used only on F, for E2). No score is kept.

In [ ]:
def ngram_fit(bodies, order):
    # tables[k][context of length k] -> {next char: count}
    tables = [dict() for _ in range(order)]
    for body in bodies:
        for i, ch in enumerate(body):
            for k in range(min(order, i + 1)):
                nxt = tables[k].setdefault(body[i - k:i], {})
                nxt[ch] = nxt.get(ch, 0) + 1
    # per context: (next-char dict, total count, distinct next chars)
    return [{h: (n, sum(n.values()), len(n)) for h, n in t.items()}
            for t in tables]


def ngram_nll(stats, body, vocab_size):
    nll = 0.0
    for i, ch in enumerate(body):
        p = 1.0 / vocab_size
        for k in range(min(len(stats), i + 1)):
            entry = stats[k].get(body[i - k:i])
            if entry is None:
                break  # longer contexts are unseen too
            n, total, types = entry
            p = (n.get(ch, 0) + types * p) / (total + types)
        nll -= math.log(p)
    return nll


t0 = time.perf_counter()
ngram_stats = ngram_fit([d.body for d in t_docs], NGRAM_ORDER)
ngram_fit_s = time.perf_counter() - t0
V = len({c for d in t_docs for c in d.body})
t0 = time.perf_counter()
_ = [ngram_nll(ngram_stats, d.body, V) for d in score_docs]
ngram_score_s = time.perf_counter() - t0
del ngram_stats, _
ngram_score_f_s = ngram_score_s / score_chars * f_chars
ngram = {"estimate": True, "reason": "timing stub; real n-gram not written",
         "order": NGRAM_ORDER, "fit_chars": t_chars, "fit_s": ngram_fit_s,
         "score_chars": score_chars, "score_s": ngram_score_s,
         "projected_to": "F", "projected_f_score_s": ngram_score_f_s,
         "total_s": ngram_fit_s + ngram_score_f_s}
print(f"n-gram (ESTIMATE, stub): fit {ngram_fit_s:.1f} s on {t_chars:,} chars; "
      f"score {ngram_score_s:.1f} s on {score_chars:,} chars "
      f"-> {ngram_score_f_s:.1f} s projected for F")

## 3e. Bootstrap (§6): timing stub

**Estimate.** The real analysis code is not written yet. This stub does the §6 work on T-sized arrays: 2,000 percentile-bootstrap resamples of meetings for each of H1, H2 and H3, with a fixed seed. Values are random placeholders (no model output). Stand-ins: H1 = difference of mean document values between the later and earlier half of T meetings, each half resampled separately; H2 = mean within-meeting (minutes − statement) difference over T meetings that have both; H3 = Spearman ρ between two per-document arrays over all T documents.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
t_meetings = sorted({d.meeting for d in t_docs})
doc_idx = {m: [i for i, d in enumerate(t_docs) if d.meeting == m]
           for m in t_meetings}
fake_a = rng.normal(2.0, 0.2, len(t_docs))  # placeholder per-document values
fake_b = fake_a + rng.normal(0.0, 0.1, len(t_docs))
genre = np.array([d.genre for d in t_docs])
half = len(t_meetings) // 2
early, late = t_meetings[:half], t_meetings[half:]
matched = [m for m in t_meetings
           if {t_docs[i].genre for i in doc_idx[m]} == {"statement", "minutes"}]
pair_diff = np.array([
    fake_a[[i for i in doc_idx[m] if genre[i] == "minutes"]].mean()
    - fake_a[[i for i in doc_idx[m] if genre[i] == "statement"]].mean()
    for m in matched])


def gather(meetings, draw):
    return np.concatenate([doc_idx[meetings[j]] for j in draw])


def rank(x):
    r = np.empty(len(x))
    r[np.argsort(x)] = np.arange(len(x))
    return r


def spearman(x, y):
    return float(np.corrcoef(rank(x), rank(y))[0, 1])


def bootstrap(stat, n):
    boot_rng = np.random.default_rng(1)  # §6: seed fixed
    vals = np.array([stat(boot_rng) for _ in range(n)])
    return np.percentile(vals, [2.5, 97.5])


BOOT_STATS = {
    "H1": lambda g: (fake_a[gather(late, g.integers(0, len(late), len(late)))].mean()
                     - fake_a[gather(early, g.integers(0, len(early), len(early)))].mean()),
    "H2": lambda g: pair_diff[g.integers(0, len(matched), len(matched))].mean(),
    "H3": lambda g: spearman(*(lambda ix: (fake_a[ix], fake_b[ix]))(
        gather(t_meetings, g.integers(0, len(t_meetings), len(t_meetings))))),
}
assert tuple(BOOT_STATS) == BOOT_HYPOTHESES
t0 = time.perf_counter()
for h in BOOT_HYPOTHESES:
    bootstrap(BOOT_STATS[h], BOOT_RESAMPLES)
bootstrap_s = time.perf_counter() - t0
boot = {"estimate": True, "reason": "timing stub; real analysis not written",
        "resamples": BOOT_RESAMPLES, "hypotheses": list(BOOT_HYPOTHESES),
        "t_meetings": len(t_meetings), "t_matched_meetings": len(matched),
        "t_documents": len(t_docs), "s": bootstrap_s}
print(f"bootstrap (ESTIMATE, stub): {BOOT_RESAMPLES:,} resamples x "
      f"{len(BOOT_HYPOTHESES)} hypotheses on {len(t_meetings)} T meetings "
      f"({len(matched)} matched): {bootstrap_s:.1f} s")

## 3f. Fixed overheads

Imports with CUDA init (cell 2), clone, load and tokenization are measured. The clone counts only when this session cloned. The checks proxy is the repo's pytest suite, run in a subprocess. Figures do not exist yet, so they use the 30 s estimate, as does any item that could not be measured. Tokenizing T for training is added because the real run pays it.

In [ ]:
t0 = time.perf_counter()
proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p",
                       "no:cacheprovider", "tests"], cwd=W3,
                      capture_output=True, text=True)
checks_s = time.perf_counter() - t0
print(proc.stdout.strip().splitlines()[-1] if proc.stdout.strip() else "")
if proc.returncode != 0:
    print(proc.stdout[-3000:], proc.stderr[-3000:])
    raise RuntimeError("test suite failed; fix before using this pilot")


def item(value):
    if value is None:
        return {"s": FIGURES_ESTIMATE_S, "measured": False}
    return {"s": value, "measured": True}


overheads = {"imports_and_cuda_init": item(import_cuda_s),
             "clone": item(clone_s), "load_and_hash_checks": item(load_s),
             "tests_equivalent_checks": item(checks_s),
             "tokenize_T_for_training": item(tokenize_s),
             "figures": item(None)}
overheads_total_s = sum(o["s"] for o in overheads.values())
for k, o in overheads.items():
    print(f"{k:>24}: {o['s']:6.1f} s {'' if o['measured'] else '(estimate)'}")
print(f"{'total':>24}: {overheads_total_s:6.1f} s")

## 4. Selection rule

In [ ]:
RULE = f'''\
DESIGN.md v0.5 section 8, selection rule (pre-registered):
Budget = {BUDGET_S:.0f} s. Subtract, all measured on T only: overheads
(imports and CUDA init, clone, load and hash checks, checks, tokenization,
figures), BPE fit, n-gram fit, char scoring projected to N+F, BPE and n-gram
scoring projected to F, bootstrap, and N monitoring.
Split the remainder equally between the char and BPE seed-1 runs.
Candidates in the order C1, C2, C3, fp16 before fp32 within each; the first
where both runs get >= {MIN_STEPS:,} steps wins, with
steps = floor(per-run budget / median s/step) (the minimum of the two).
{VOCAB_RULE} (Applied in 3a.)
Seed 2 (char only) is added only if one more char run at the same config and
steps, plus its monitoring and F scoring, fits in the budget left after the
selection. It never changes config or steps.
If nothing qualifies: NO CONFIG FITS (study infeasible under section 10).'''
print(RULE, "\n")
print(f"BPE vocab: {BPE_VOCAB or 'none qualifies'}")

candidates, choice = [], None
if BPE_VOCAB is not None:
    bpe_fit_s = bpe_stats[BPE_VOCAB]["fit_s"]
    for c in CONFIGS:
        score_s = (scoring[f"{c}/char"]["projected_s"]  # N+F
                   + scoring[f"{c}/bpe"]["projected_s"]  # F
                   + ngram["projected_f_score_s"])
        for p in PRECISIONS:
            t_char = throughput[f"{c}/char/{p}"]
            t_bpe = throughput[f"{c}/bpe/{p}"]
            monitor_s = t_char["monitor_run_s"] + t_bpe["monitor_run_s"]
            remainder = (BUDGET_S - overheads_total_s - bpe_fit_s
                         - ngram["fit_s"] - score_s - bootstrap_s - monitor_s)
            per_run = remainder / 2
            s_char = t_char["s_per_step_median"]
            s_bpe = t_bpe["s_per_step_median"]
            steps = max(0, min(math.floor(per_run / s_char),
                               math.floor(per_run / s_bpe)))
            row = {"config": c, "precision": p, "scoring_s": score_s,
                   "monitoring_s": monitor_s, "remainder_s": remainder,
                   "per_run_s": per_run, "s_per_step_char": s_char,
                   "s_per_step_bpe": s_bpe, "steps": steps,
                   "qualifies": steps >= MIN_STEPS}
            candidates.append(row)
            if choice is None and row["qualifies"]:
                choice = row

print(f"{'cfg':>4} {'prec':>5} {'scoring s':>10} {'monitor s':>10} "
      f"{'remainder s':>12} {'per run s':>10} {'ms/step char':>13} "
      f"{'ms/step bpe':>12} {'steps':>7}  ok")
for r in candidates:
    print(f"{r['config']:>4} {r['precision']:>5} {r['scoring_s']:10.1f} "
          f"{r['monitoring_s']:10.1f} {r['remainder_s']:12.1f} "
          f"{r['per_run_s']:10.1f} {r['s_per_step_char']*1e3:13.1f} "
          f"{r['s_per_step_bpe']*1e3:12.1f} {r['steps']:7d}  "
          f"{'yes' if r['qualifies'] else 'no'}")

print()
if choice is None:
    selection = {"result": "NO CONFIG FITS", "bpe_vocab": BPE_VOCAB}
    seed2 = {"added": False, "reason": "no config fits"}
    print("NO CONFIG FITS: study infeasible under DESIGN.md section 10")
else:
    # Seed 2: one more char run at the chosen config and steps, plus its
    # N monitoring and F scoring, must fit in what the selection left.
    steps = choice["steps"]
    key = f"{choice['config']}/char/{choice['precision']}"
    left = choice["remainder_s"] - steps * (choice["s_per_step_char"]
                                            + choice["s_per_step_bpe"])
    seed2_train = steps * choice["s_per_step_char"]
    seed2_monitor = throughput[key]["monitor_run_s"]
    seed2_score_f = scoring[f"{choice['config']}/char"]["projected_f_s"]
    seed2_cost = seed2_train + seed2_monitor + seed2_score_f
    seed2 = {"added": seed2_cost <= left, "budget_left_s": left,
             "cost_s": seed2_cost, "train_s": seed2_train,
             "monitoring_s": seed2_monitor, "f_scoring_s": seed2_score_f,
             "config": choice["config"], "precision": choice["precision"],
             "steps": steps}
    selection = {"result": "OK", "config": choice["config"],
                 **CONFIGS[choice["config"]], "precision": choice["precision"],
                 "block_size": BLOCK_SIZE, "batch": BATCH, "steps": steps,
                 "bpe_vocab": BPE_VOCAB, "char_seed2": seed2["added"]}
    print("RESULT: " + ", ".join(f"{k}={v}" for k, v in selection.items()
                                 if k != "result"))
    print(f"seed 2: needs {seed2_cost:.1f} s (train {seed2_train:.1f} + "
          f"monitoring {seed2_monitor:.1f} + F scoring {seed2_score_f:.1f}); "
          f"left {left:.1f} s -> {'ADDED' if seed2['added'] else 'not added'}")
if SMOKE:
    print("(SMOKE RUN: numbers are meaningless; do not use this result)")

## 5. Save measurements

In [ ]:
import json

record = {
    "design": "DESIGN.md v0.5 section 8, Phase 3 pilot",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": ENV,
    "settings": {"block_size": BLOCK_SIZE, "batch": BATCH,
                 "warmup_steps": WARMUP_STEPS, "timed_steps": TIMED_STEPS,
                 "configs": CONFIGS, "precisions": list(PRECISIONS),
                 "monitoring": {"evals_per_run": MON_EVALS,
                                "batches_per_eval": MON_BATCHES,
                                "windows_per_batch": MON_BATCH,
                                "warmup_batches": MON_WARMUP,
                                "timed_on": "T windows"},
                 "bootstrap": {"resamples": BOOT_RESAMPLES,
                               "hypotheses": list(BOOT_HYPOTHESES)},
                 "budget_s": BUDGET_S, "min_steps": MIN_STEPS,
                 "bpe_fit_limit_s": BPE_FIT_LIMIT_S},
    "data": {"t_documents": len(t_docs), "t_chars": t_chars,
             "nf_chars_from_manifest": nf_chars,
             "f_chars_from_manifest": f_chars},
    "bpe": {str(v): s for v, s in bpe_stats.items()},
    "bpe_vocab": BPE_VOCAB, "bpe_vocab_rule": VOCAB_RULE,
    "training_throughput": throughput,
    "monitoring_run_s": {k: r["monitor_run_s"] for k, r in throughput.items()},
    "scoring": scoring,
    "ngram": ngram,
    "bootstrap": boot,
    "overheads": overheads, "overheads_total_s": overheads_total_s,
    "rule": RULE, "candidates": candidates, "selection": selection, "seed2": seed2,
}
OUT_PATH.write_text(json.dumps(record, indent=2) + "\n")
print(f"saved {OUT_PATH}\n")
print(json.dumps(record, indent=2))